# 03. 基于内容的推荐算法 (Content-Based Recommendation)

本 Notebook 是 [03_基于内容的推荐.md](03_基于内容的推荐.md) 的配套代码。

我们将实现：
1. **手工计算验证**：使用 NumPy 验证文档中的手工计算案例。
2. **实战 Pipeline**：基于 TF-IDF 和 Cosine Similarity 构建一个简单的电影推荐系统。
   - 场景 A：Item-to-Item（看这部电影的人也喜欢...）
   - 场景 B：User-to-Item（基于用户画像的个性化推荐）

In [13]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

## 1. 手工计算示例验证

对应文档章节：**2. 示例计算：手工算一次“内容推荐”**

我们验证文档中的计算结果：
- 用户喜欢电影 A [1, 1, 0] 和 电影 B [0, 1, 1]
- 候选电影 C [1, 0, 1]
- 计算用户画像与电影 C 的余弦相似度

In [14]:
# 1. 定义电影向量
movie_a = np.array([1, 1, 0]) # Action, Comedy
movie_b = np.array([0, 1, 1]) # Comedy, Romance
movie_c = np.array([1, 0, 1]) # Action, Romance (Candidate)

# 2. 构建用户画像 (平均法)
user_profile = (movie_a + movie_b) / 2
print(f"用户画像向量 (User Profile Vector): {user_profile}")

# 3. 计算余弦相似度
# 公式: dot(u, c) / (norm(u) * norm(c))

# 手动计算
dot_product = np.dot(user_profile, movie_c)
norm_user = np.linalg.norm(user_profile)
norm_c = np.linalg.norm(movie_c)
similarity_manual = dot_product / (norm_user * norm_c)

print(f"点积 (Dot Product): {dot_product}")
print(f"用户向量模长 (Norm User): {norm_user:.4f}")
print(f"物品 C 向量模长 (Norm C): {norm_c:.4f}")
print(f"相似度 (手工计算): {similarity_manual:.4f}")

# 使用 sklearn 验证
# reshape(1, -1) 是因为 sklearn 期望二维数组
similarity_sklearn = cosine_similarity(user_profile.reshape(1, -1), movie_c.reshape(1, -1))[0][0]
print(f"相似度 (Sklearn): {similarity_sklearn:.4f}")

用户画像向量 (User Profile Vector): [0.5 1.  0.5]
点积 (Dot Product): 1.0
用户向量模长 (Norm User): 1.2247
物品 C 向量模长 (Norm C): 1.4142
相似度 (手工计算): 0.5774
相似度 (Sklearn): 0.5774


## 2. 工程实践：基于 TF-IDF 的电影推荐 Pipeline

对应文档章节：**3. 工程实践：基于 TF-IDF 的电影推荐 Pipeline**

In [9]:
# 1. 模拟数据
data = {
    'id': [1, 2, 3, 4, 5],
    'title': ['The Matrix', 'John Wick', 'Toy Story', 'Finding Nemo', 'Interstellar'],
    'plot': [
        'A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.',
        'An ex-hitman comes out of retirement to track down the gangsters that killed his dog and took everything from him.',
        'A cowboy doll is profoundly threatened and jealous when a new spaceman figure supplants him as top toy in a boy\'s room.',
        'After his son is captured in the Great Barrier Reef and taken to Sydney, a timid clownfish sets out on a journey to bring him home.',
        'A team of explorers travel through a wormhole in space in an attempt to ensure humanity\'s survival.'
    ]
}

df = pd.DataFrame(data)
df

,id,title,plot
0,1,The Matrix,A computer hacker learns from mysterious rebel...
1,2,John Wick,An ex-hitman comes out of retirement to track ...
2,3,Toy Story,A cowboy doll is profoundly threatened and jea...
3,4,Finding Nemo,After his son is captured in the Great Barrier...
4,5,Interstellar,A team of explorers travel through a wormhole ...


In [10]:
# 2. 特征工程 Pipeline
tfidf = TfidfVectorizer(stop_words='english')

# 构建物品画像矩阵 (Item Profile Matrix)
tfidf_matrix = tfidf.fit_transform(df['plot'])

print("TF-IDF 矩阵形状:", tfidf_matrix.shape)
print("特征名称 (词汇表):", tfidf.get_feature_names_out())

TF-IDF 矩阵形状: (5, 54)
特征名称 (词汇表): ['attempt' 'barrier' 'boy' 'bring' 'captured' 'clownfish' 'comes'
 'computer' 'controllers' 'cowboy' 'dog' 'doll' 'ensure' 'ex' 'explorers'
 'figure' 'gangsters' 'great' 'hacker' 'hitman' 'home' 'humanity'
 'jealous' 'journey' 'killed' 'learns' 'mysterious' 'nature' 'new'
 'profoundly' 'reality' 'rebels' 'reef' 'retirement' 'role' 'room' 'sets'
 'son' 'space' 'spaceman' 'supplants' 'survival' 'sydney' 'taken' 'team'
 'threatened' 'timid' 'took' 'toy' 'track' 'travel' 'true' 'war'
 'wormhole']


In [11]:
class ContentBasedRecommender:
    def __init__(self, item_vectors, items_df):
        self.item_vectors = item_vectors
        self.items_df = items_df

    def get_recommendations(self, item_title, top_k=2):
        """
        场景 A：Item-to-Item 推荐
        输入用户喜欢的一部电影，推荐相似电影
        """
        # 1. 找到输入电影的索引
        try:
            idx = self.items_df.index[self.items_df['title'] == item_title].tolist()[0]
        except IndexError:
            return "未找到该物品。"

        # 2. 计算该电影与所有其他电影的余弦相似度
        # linear_kernel 计算点积。由于 TF-IDF 向量通常已 L2 归一化，点积等同于余弦相似度
        cosine_sim = linear_kernel(self.item_vectors[idx:idx+1], self.item_vectors).flatten()

        # 3. 获取 Top-K 索引 (排除自身)
        sim_scores = list(enumerate(cosine_sim))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # [1:top_k+1] 排除自身 (index 0 usually)
        sim_indices = [i[0] for i in sim_scores[1:top_k+1]]

        return self.items_df.iloc[sim_indices][['title', 'plot']]

    def get_user_recommendations(self, user_liked_titles, top_k=2):
        """
        场景 B：User-to-Item 推荐
        基于用户历史喜欢的电影列表，构建用户画像向量，推荐电影
        """
        # 1. 获取用户喜欢的电影的索引
        indices = self.items_df.index[self.items_df['title'].isin(user_liked_titles)].tolist()
        
        if not indices:
            return "历史记录中未找到有效物品。"

        # 2. 构建用户画像向量 (User Profile Vector)
        # 策略：计算用户喜欢的电影向量的平均值
        # 注意：这里 item_vectors 是稀疏矩阵，需要先转为 dense 或者直接操作
        # scipy sparse matrix mean returns a matrix, need to convert to array
        user_profile = self.item_vectors[indices].mean(axis=0)
        # mean 返回的是 matrix 类型 (1, n_features)，需要转为 array 以便后续计算一致性
        user_profile = np.asarray(user_profile)

        # 3. 计算用户向量与所有物品向量的相似度
        cosine_sim = linear_kernel(user_profile, self.item_vectors).flatten()

        # 4. 获取 Top-K 结果 (排除用户已经看过的电影)
        sim_scores = list(enumerate(cosine_sim))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # 过滤掉已看过的
        recommendations = []
        for i, score in sim_scores:
            if i not in indices:
                recommendations.append(i)
            if len(recommendations) >= top_k:
                break
        
        return self.items_df.iloc[recommendations][['title', 'plot']]

# 初始化推荐服务
recommender = ContentBasedRecommender(tfidf_matrix, df)

In [12]:
# 测试 1：Item-to-Item
print("--- 场景 A: 基于 'The Matrix' 的相似推荐 ---")
print(recommender.get_recommendations('The Matrix'))

print("\n" + "="*50 + "\n")

# 测试 2：User-to-Item
# 假设用户喜欢 'The Matrix' (Action/Sci-Fi) 和 'Interstellar' (Sci-Fi/Adventure)
print("--- 场景 B: 基于用户画像 (喜欢 Matrix + Interstellar) 的推荐 ---")
print(recommender.get_user_recommendations(['The Matrix', 'Interstellar']))

--- 场景 A: 基于 'The Matrix' 的相似推荐 ---
       title                                               plot
1  John Wick  An ex-hitman comes out of retirement to track ...
2  Toy Story  A cowboy doll is profoundly threatened and jea...


--- 场景 B: 基于用户画像 (喜欢 Matrix + Interstellar) 的推荐 ---
       title                                               plot
1  John Wick  An ex-hitman comes out of retirement to track ...
2  Toy Story  A cowboy doll is profoundly threatened and jea...
